# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EgeGln365/FlyRank_AI_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

#### Finding 1 — The Anatomy of Growing Content

The paper found that content with growing impressions tends to be longer and younger than declining content. Growing pages had around 3,180 words and an average age of 184 days, while declining pages had around 2,311 words and an average age of 230 days.

The growth label comes from the change in impressions between the latest 30 days and the previous 30 days. Pages with more than 10% growth are labeled as "up", while pages with more than 10% decline are labeled as "down".

**Methodology question:** Would the relationship between content age, word count, and growth remain similar if the trend label were measured over a longer time window or validated on a later period?

The comparison is useful for identifying patterns in the portfolio, but it is observational. Therefore, it shows an association between these features and growth rather than proving that increasing word count or having newer content directly causes growth.

#### Finding 3 — Click Capture by Position Tier

The paper found that weighted CTR decreases as content moves to lower search position tiers. The Top 3 positions had a weighted CTR of 0.423%, while Page 1 positions (4–10) had 0.339%, positions 11–20 had 0.325%, positions 21–50 had 0.163%, and positions above 50 had only 0.050%.

This finding does not use a classification label like the growing/declining label in Finding 1. Instead, the measured outcome is weighted CTR, which is calculated as total clicks divided by total impressions within each search position tier.

**Methodology question:** Would the same decline in CTR across position tiers remain if the analysis were repeated within individual clients or using a client-grouped analysis, rather than only using portfolio-level aggregated results?

The comparison shows a clear relationship between search position and CTR in this portfolio. However, because the results are aggregated across many brands and content pages, some high-traffic clients or pages may have a larger influence on the weighted CTR. A client-grouped analysis could help check whether the same pattern is consistent across different clients.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")


if HF_TOKEN is None:
    raise ValueError("HF_TOKEN bulunamadı. .env dosyanı kontrol et.")


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [5]:
march_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days_march,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_march,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS ctr_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_march

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2026-03-01'
                      AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [6]:
april_content = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_april,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_april,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS ctr_april,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_april

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2026-04-01'
                      AND DATE '2026-04-30'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [9]:
modeling_frame = march_features.merge(
    april_content,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_data = modeling_frame[
    (modeling_frame["gsc_available_days_march"] >= 20)
    &
    (modeling_frame["gsc_available_days"] >= 20)
].copy()

print("Modeling rows:", len(model_data))

Modeling rows: 95633


In [10]:
april_good_position = model_data[
    (model_data["avg_position_april"] > 0)
    &
    (model_data["avg_position_april"] <= 10)
    &
    (model_data["ctr_april"] > 0)
].copy()

ctr_threshold_april = (
    april_good_position["ctr_april"].quantile(0.25)
)

print("April CTR threshold:", ctr_threshold_april)

April CTR threshold: 0.001394335515512089


In [11]:
opportunity_mask = (
    (model_data["avg_position_april"] > 0)
    &
    (model_data["avg_position_april"] <= 10)
    &
    (model_data["impressions_april"] >= 500)
    &
    (model_data["ctr_april"] <= ctr_threshold_april)
)

model_data["opportunity"] = opportunity_mask.astype(int)

print("Positive opportunities:", model_data["opportunity"].sum())
print("Total contents:", len(model_data))
print("Base rate:", model_data["opportunity"].mean())

Positive opportunities: 12273
Total contents: 95633
Base rate: 0.12833436156975103


In [12]:
prev90_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days_prev90,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_prev90,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_prev90,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
    END AS ctr_prev90,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
    END AS avg_position_prev90

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2025-12-01'
                      AND DATE '2026-02-28'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [13]:
model_with_history = model_data.merge(
    prev90_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Model data rows:", len(model_data))
print("After history join:", len(model_with_history))

Model data rows: 95633
After history join: 95633


In [14]:
model_with_history["impressions_per_day_march"] = (
    model_with_history["impressions_march"]
    / model_with_history["gsc_available_days_march"]
)

model_with_history["impressions_per_day_prev90"] = (
    model_with_history["impressions_prev90"]
    / model_with_history["gsc_available_days_prev90"]
)

model_with_history["ctr_change"] = (
    model_with_history["ctr_march"]
    - model_with_history["ctr_prev90"]
)

model_with_history["position_change"] = (
    model_with_history["avg_position_march"]
    - model_with_history["avg_position_prev90"]
)

model_with_history["impressions_per_day_change"] = (
    model_with_history["impressions_per_day_march"]
    - model_with_history["impressions_per_day_prev90"]
)

model_with_history["has_prev90_history"] = (
    model_with_history["gsc_available_days_prev90"]
    .fillna(0)
    .gt(0)
    .astype(int)
)

In [15]:
final_features = [
    # Current state
    "impressions_per_day_march",
    "ctr_march",
    "avg_position_march",

    # Historical state
    "impressions_per_day_prev90",
    "ctr_prev90",
    "avg_position_prev90",

    # Trend
    "ctr_change",
    "position_change",
    "impressions_per_day_change",

    # Historical data coverage
    "gsc_available_days_prev90",
    "has_prev90_history",
]

target = "opportunity"

X = model_with_history[final_features].copy()
y = model_with_history[target].copy()
groups = model_with_history["client_hash_id"].copy()

print("Rows:", len(X))
print("Features:", X.shape[1])
print("Clients:", groups.nunique())
print("Positive rate:", y.mean())

Rows: 95633
Features: 11
Clients: 37
Positive rate: 0.12833436156975103


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

In [17]:
import numpy as np

def precision_at_k(y_true, y_score, k):
    order = np.argsort(y_score)[::-1]
    top_k_idx = order[:k]

    top_k_true = np.asarray(y_true)[top_k_idx]

    return top_k_true.mean()

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import pandas as pd

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

before_results = []

for fold, (train_idx, test_idx) in enumerate(
    skf.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])

    client_overlap = len(
        train_clients.intersection(test_clients)
    )

    fold_model = clone(logreg_pipeline)

    fold_model.fit(X_train, y_train)

    y_prob = fold_model.predict_proba(X_test)[:, 1]

    p20 = precision_at_k(y_test, y_prob, 20)
    p50 = precision_at_k(y_test, y_prob, 50)

    before_results.append({
        "fold": fold,
        "p20": p20,
        "p50": p50,
        "client_overlap": client_overlap,
        "test_positive_rate": y_test.mean()
    })

before_results = pd.DataFrame(before_results)

before_results

,fold,p20,p50,client_overlap,test_positive_rate
0,1,0.40,0.58,33,0.128353
1,2,0.40,0.50,36,0.128353
2,3,0.45,0.60,34,0.128353
3,4,0.50,0.62,33,0.128307
4,5,0.60,0.60,32,0.128307


In [19]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

after_results = []

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])

    client_overlap = len(
        train_clients.intersection(test_clients)
    )

    fold_model = clone(logreg_pipeline)

    fold_model.fit(X_train, y_train)

    y_prob = fold_model.predict_proba(X_test)[:, 1]

    p20 = precision_at_k(y_test, y_prob, 20)
    p50 = precision_at_k(y_test, y_prob, 50)

    after_results.append({
        "fold": fold,
        "p20": p20,
        "p50": p50,
        "client_overlap": client_overlap,
        "test_positive_rate": y_test.mean()
    })

after_results = pd.DataFrame(after_results)

after_results

,fold,p20,p50,client_overlap,test_positive_rate
0,1,0.45,0.32,0,0.193678
1,2,0.60,0.66,0,0.068759
2,3,0.55,0.62,0,0.187581
3,4,0.40,0.40,0,0.081767
4,5,0.45,0.58,0,0.090739


In [20]:
validation_comparison = pd.DataFrame({
    "validation": [
        "Before: StratifiedKFold",
        "After: StratifiedGroupKFold"
    ],

    "mean_p20": [
        before_results["p20"].mean(),
        after_results["p20"].mean()
    ],

    "mean_p50": [
        before_results["p50"].mean(),
        after_results["p50"].mean()
    ],

    "mean_client_overlap": [
        before_results["client_overlap"].mean(),
        after_results["client_overlap"].mean()
    ],

    "std_p20": [
        before_results["p20"].std(),
        after_results["p20"].std()
    ],

    "std_p50": [
        before_results["p50"].std(),
        after_results["p50"].std()
    ]
})

validation_comparison

,validation,mean_p20,mean_p50,mean_client_overlap,std_p20,std_p50
0,Before: StratifiedKFold,0.47,0.580,33.6,0.083666,0.046904
1,After: StratifiedGroupKFold,0.49,0.516,0.0,0.082158,0.147919


### Validation comparison

The naive row-level StratifiedKFold split produced substantial client overlap between training and test data, with an average of 33.6 clients appearing in both sets. In contrast, StratifiedGroupKFold reduced client overlap to zero, ensuring that evaluation was performed on unseen clients.

The change in validation design did not affect all ranking metrics in the same direction. Mean Precision@20 increased slightly from 47.0% to 49.0%, while mean Precision@50 decreased from 58.0% to 51.6%. The grouped evaluation also showed greater variation across folds, particularly for Precision@50.

These results show that the measured performance is sensitive to the validation design and to which clients are held out. The grouped result is more appropriate for evaluating generalization to unseen clients because content from the same client cannot appear in both training and test sets.

Importantly, the Week-5 model already used the grouped validation design. The row-level split is included here only as a comparison to audit the effect of enforcing client separation.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [21]:
# Audit the final model features for obvious leakage risks

audit_rows = []

for feature in final_features:

    if feature.endswith("_march"):
        source_window = "March 2026"
    elif "prev90" in feature:
        source_window = "Dec 2025 - Feb 2026"
    elif feature in [
        "ctr_change",
        "position_change",
        "impressions_per_day_change"
    ]:
        source_window = "Prev90 + March"
    else:
        source_window = "Check manually"

    audit_rows.append({
        "feature": feature,
        "source_window": source_window,
        "contains_april": "april" in feature.lower(),
        "is_identifier": feature in [
            "client_hash_id",
            "content_hash_id"
        ],
        "is_target": feature == target
    })

leakage_audit = pd.DataFrame(audit_rows)

leakage_audit

,feature,source_window,contains_april,is_identifier,is_target
0,impressions_per_day_march,March 2026,False,False,False
1,ctr_march,March 2026,False,False,False
2,avg_position_march,March 2026,False,False,False
3,impressions_per_day_prev90,Dec 2025 - Feb 2026,False,False,False
4,ctr_prev90,Dec 2025 - Feb 2026,False,False,False
5,avg_position_prev90,Dec 2025 - Feb 2026,False,False,False
6,ctr_change,Prev90 + March,False,False,False
7,position_change,Prev90 + March,False,False,False
8,impressions_per_day_change,Prev90 + March,False,False,False
9,gsc_available_days_prev90,Dec 2025 - Feb 2026,False,False,False


In [22]:
print(
    "Features containing April information:",
    leakage_audit["contains_april"].sum()
)

print(
    "Identifier features:",
    leakage_audit["is_identifier"].sum()
)

print(
    "Target included as feature:",
    leakage_audit["is_target"].sum()
)

Features containing April information: 0
Identifier features: 0
Target included as feature: 0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim:**

> Logistic Regression captured additional patterns not directly represented by the rule-based baseline.

**Rewritten claim:**

> Logistic Regression ranked some true future opportunities in its Top 50 that were not ranked in the baseline Top 50. Their feature profiles suggest that the model may be responding to patterns beyond the baseline's explicit high-visibility, low-CTR rule.

This wording treats the result as directional evidence rather than proof of a specific model mechanism.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.